#RL Actor Critic PPO Policy/Agent for Motion Planning end to end in MetaDrive Simulator/Environment

You will:
- Formulate autonomous driving as an MDP
- Train a PPO agent (Stable-Baselines version)
- Evaluate generalization
- Deploy and visualize the policy
- Launch a Gradio demo


In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Dynamically add src/ to Python path
notebook_dir = os.getcwd()
if 'notebooks' in notebook_dir:
    project_root = os.path.dirname(notebook_dir)
else:
    project_root = notebook_dir

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Verify src/ exists
if not os.path.exists(src_path):
    raise FileNotFoundError(f"❌ src/ folder not found at: {src_path}")

# Import our modular packages
from env_config import get_env_config
from train import create_ppo_model, train_model
from evaluate import evaluate_policy, run_benchmark
from visualize import generate_demo_suite, create_gradio_demo

# Import third-party
from stable_baselines3 import PPO
import gymnasium as gym

print("✅ Step 1 Complete: All modules imported successfully!")
print(f"   Project root: {project_root}")
print(f"   Src path: {src_path}")

✅ Step 1 Complete: All modules imported successfully!
   Project root: /home/vijay2998/meta-drive-ppo-highway-planner
   Src path: /home/vijay2998/meta-drive-ppo-highway-planner/src


In [2]:
!pip install -q git+https://github.com/metadriverse/metadrive.git
!pip install -q stable-baselines3 gymnasium torch gradio imageio

In [3]:
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'

In [4]:
import numpy as np
import torch
import imageio
import gradio as gr

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

from metadrive import MetaDriveEnv
from metadrive.engine.engine_utils import close_engine

close_engine()

In [5]:
ENV_CONFIG = {
    "use_render": False,
    "num_scenarios": 50,
    "start_seed": 0,
    "traffic_density": 0.1,
    "random_lane_width": True,
    "random_agent_model": True,
    "num_agents": 1,
    "horizon": 1000,

    # 🔴 CRITICAL: freeze sensors
    "sensors": {
        "lidar": (),
        "side_detector": (),
        "lane_line_detector": ()
    }
}


## 1. MetaDrive Environment (Single Agent, Safe Setup)

In [6]:
close_engine()

env = MetaDriveEnv(ENV_CONFIG)
env = Monitor(env)

print('Observation space:', env.observation_space)
print('Action space:', env.action_space)

[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000


Observation space: Box(-0.0, 1.0, (261,), float32)
Action space: Box(-1.0, 1.0, (2,), float32)


## 2. PPO Actor–Critic Configuration

In [7]:
print("Initializing PPO agent with modular configuration...")

# Get training configuration from our module
from env_config import get_env_config
env_config = get_env_config("train")

# Create model and environment using our modular function
from train import create_ppo_model
model, env = create_ppo_model(env_config, verbose=1)

print("✅ PPO agent initialized successfully!")
print(f"   Observation space: {env.observation_space}")
print(f"   Action space: {env.action_space}")

[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000


Initializing PPO agent with modular configuration...
⚠️ close_engine not found, skipping...
Observation space: Box(-0.0, 1.0, (261,), float32)
Action space: Box(-1.0, 1.0, (2,), float32)
Using cpu device
Wrapping the env in a DummyVecEnv.
✅ PPO agent initialized successfully!
   Observation space: Box(-0.0, 1.0, (261,), float32)
   Action space: Box(-1.0, 1.0, (2,), float32)


## 3. Train PPO Agent

In [8]:
print("Starting training...")
print("This will take ~10-15 minutes on CPU, ~5 minutes on GPU.")

# Train using our modular function
from train import train_model

model = train_model(
    model,
    total_timesteps=100000,  # 100k steps
    save_path="models/ppo_metadrive_safe"
)

# Close the environment
env.close()

print("\n✅ Training complete!")
print(f"   Model saved to: models/ppo_metadrive_safe.zip")

[WARNING] Assets folder doesn't exist. Begin to download assets... (base_engine.py:773)
[INFO] Pull assets from https://github.com/metadriverse/metadrive/releases/download/MetaDrive-0.4.3/assets.zip to /home/vijay2998/meta-drive-ppo-highway-planner/.venv/lib/python3.11/site-packages/metadrive/assets.zip


Starting training...
This will take ~10-15 minutes on CPU, ~5 minutes on GPU.
Training for 100000 timesteps...


100% |########################################################################|
[INFO] Extracting assets.
[INFO] Successfully download assets, version: 0.4.3. MetaDrive version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe
[INFO] Start Scenario Index: 0, Num Scenarios : 50
[INFO] Episode ended! Scenario Index: 25 Reason: max step 
[INFO] Episode ended! Scenario Index: 17 Reason: max step 


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | 11.5     |
| time/              |          |
|    fps             | 91       |
|    iterations      | 1        |
|    time_elapsed    | 22       |
|    total_timesteps | 2048     |
---------------------------------


[INFO] Episode ended! Scenario Index: 3 Reason: max step 
[INFO] Episode ended! Scenario Index: 7 Reason: out_of_road.


...

----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 143        |
|    ep_rew_mean          | 43.3       |
| time/                   |            |
|    fps                  | 183        |
|    iterations           | 48         |
|    time_elapsed         | 536        |
|    total_timesteps      | 98304      |
| train/                  |            |
|    approx_kl            | 0.11530061 |
|    clip_fraction        | 0.299      |
|    clip_range           | 0.2        |
|    entropy_loss         | -2.18      |
|    explained_variance   | 0.4495762  |
|    learning_rate        | 0.0003     |
|    loss                 | 18.5       |
|    n_updates            | 470        |
|    policy_gradient_loss | -0.00666   |
|    std                  | 0.722      |
|    value_loss           | 34.5       |
----------------------------------------


[INFO] Episode ended! Scenario Index: 1 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 23 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 33 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 36 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 40 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 44 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 34 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 41 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 24 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 20 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 44 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 20 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 33 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 27 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 36 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 45 Reason: out_of_road.
[INFO] Ep

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 118         |
|    ep_rew_mean          | 33.9        |
| time/                   |             |
|    fps                  | 183         |
|    iterations           | 49          |
|    time_elapsed         | 546         |
|    total_timesteps      | 100352      |
| train/                  |             |
|    approx_kl            | 0.009353286 |
|    clip_fraction        | 0.148       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.18       |
|    explained_variance   | 0.30712318  |
|    learning_rate        | 0.0003      |
|    loss                 | 18          |
|    n_updates            | 480         |
|    policy_gradient_loss | -0.000391   |
|    std                  | 0.724       |
|    value_loss           | 36.5        |
-----------------------------------------


/home/vijay2998/meta-drive-ppo-highway-planner/.venv/lib/python3.11/site-packages/stable_baselines3/common/save_util.py:284: UserWarning: Path 'models' does not exist. Will create it.
  warnings.warn(f"Path '{path.parent}' does not exist. Will create it.")


Model saved to models/ppo_metadrive_safe

✅ Training complete!
   Model saved to: models/ppo_metadrive_safe.zip


In [9]:

print("Evaluating trained policy on unseen scenarios (seeds 1000+)...")
print("This will run 5 episodes and calculate mean reward.\n")

from evaluate import evaluate_policy, run_benchmark

# Evaluate on unseen seeds
mean_reward, std_reward = evaluate_policy(
    model,
    config_type="eval",  # Uses start_seed=1000
    episodes=5
)

print(f"\n📊 Evaluation Results:")
print(f"   Mean Reward: {mean_reward:.2f} ± {std_reward:.2f}")
print(f"   Training Reward (final): ~100 (from logs)")
print(f"   Generalization Gap: {((100 - mean_reward) / 100 * 100):.1f}%")

# Optional: Run benchmark vs Random Agent
print("\n🏃 Running benchmark: PPO vs Random Agent (3 episodes each)...")
benchmark = run_benchmark(model, random_episodes=3)

print(f"\n🏆 Benchmark Results:")
print(f"   PPO Agent:     {benchmark['ppo_mean']:.2f} ± {benchmark['ppo_std']:.2f}")
print(f"   Random Agent:  {benchmark['random_mean']:.2f} ± {benchmark['random_std']:.2f}")
if benchmark['random_mean'] > 0:
    improvement = (benchmark['ppo_mean'] / benchmark['random_mean']) * 100
    print(f"   Improvement:   {improvement:.1f}% better than random")
else:
    print(f"   Improvement:   PPO vastly outperforms random!")

print("\n✅ Step 4 Complete: Evaluation finished!")


[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe
[INFO] Start Scenario Index: 1000, Num Scenarios : 50


Evaluating trained policy on unseen scenarios (seeds 1000+)...
This will run 5 episodes and calculate mean reward.

⚠️ close_engine not found, skipping...
Evaluating on 5 episodes (eval scenarios)...


[INFO] Episode ended! Scenario Index: 1019 Reason: out_of_road.


  Episode 1: Reward = 81.11, Steps = 95


[INFO] Episode ended! Scenario Index: 1011 Reason: out_of_road.


...

  Episode 5: Reward = 132.36, Steps = 152


[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe
[INFO] Start Scenario Index: 1000, Num Scenarios : 50



Evaluation Results (eval):
  Mean Reward: 91.12 ± 29.77

📊 Evaluation Results:
   Mean Reward: 91.12 ± 29.77
   Training Reward (final): ~100 (from logs)
   Generalization Gap: 8.9%

🏃 Running benchmark: PPO vs Random Agent (3 episodes each)...
⚠️ close_engine not found, skipping...


[INFO] Episode ended! Scenario Index: 1039 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 1035 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 1028 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 1026 Reason: max step 
[INFO] Episode ended! Scenario Index: 1039 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 1010 Reason: out_of_road.



🏆 Benchmark Results:
   PPO Agent:     93.80 ± 26.67
   Random Agent:  3.64 ± 4.60
   Improvement:   2580.0% better than random

✅ Step 4 Complete: Evaluation finished!


## 4. Evaluation on Unseen Maps

In [10]:
print("Evaluating on unseen scenarios (seeds 1000+)...")

# Evaluate on unseen seeds
mean_reward, std_reward = evaluate_policy(
    model,
    config_type="eval",
    episodes=5
)

print(f"\n📊 Evaluation Results:")
print(f"   Mean Reward: {mean_reward:.2f} ± {std_reward:.2f}")

# Optional: Run benchmark vs random agent
print("\nRunning benchmark comparison (PPO vs Random Agent)...")
benchmark = run_benchmark(model, random_episodes=3)

print(f"\n🏆 Benchmark Results:")
print(f"   PPO Agent:     {benchmark['ppo_mean']:.2f} ± {benchmark['ppo_std']:.2f}")
print(f"   Random Agent:  {benchmark['random_mean']:.2f} ± {benchmark['random_std']:.2f}")
print(f"   Improvement:   {(benchmark['ppo_mean'] / max(benchmark['random_mean'], 1) * 100):.1f}% better")

print("\n✅ Step 3 Complete: Evaluation finished!")


[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe
[INFO] Start Scenario Index: 1000, Num Scenarios : 50


Evaluating on unseen scenarios (seeds 1000+)...
⚠️ close_engine not found, skipping...
Evaluating on 5 episodes (eval scenarios)...


[INFO] Episode ended! Scenario Index: 1031 Reason: out_of_road.


  Episode 1: Reward = 204.42, Steps = 179


[INFO] Episode ended! Scenario Index: 1021 Reason: out_of_road.


...

  Episode 5: Reward = 204.44, Steps = 179


[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe
[INFO] Start Scenario Index: 1000, Num Scenarios : 50



Evaluation Results (eval):
  Mean Reward: 119.29 ± 69.67

📊 Evaluation Results:
   Mean Reward: 119.29 ± 69.67

Running benchmark comparison (PPO vs Random Agent)...
⚠️ close_engine not found, skipping...


[INFO] Episode ended! Scenario Index: 1010 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 1045 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 1011 Reason: out_of_road.
[INFO] Episode ended! Scenario Index: 1006 Reason: max step 
[INFO] Episode ended! Scenario Index: 1012 Reason: max step 
[INFO] Episode ended! Scenario Index: 1017 Reason: out_of_road.



🏆 Benchmark Results:
   PPO Agent:     46.26 ± 12.76
   Random Agent:  7.24 ± 4.64
   Improvement:   638.6% better

✅ Step 3 Complete: Evaluation finished!


## 5. Policy Deployment & Visualization

In [11]:
close_engine()

render_env = MetaDriveEnv(ENV_CONFIG)

obs, _ = render_env.reset()
frames = []

for _ in range(800):
    action, _ = model.predict(obs, deterministic=True)
    obs, r, term, trunc, _ = render_env.step(action)
    frame = render_env.render(mode='top_down', screen_size=(500, 500))
    frames.append(frame)
    if term or trunc:
        break

render_env.close()
imageio.mimsave('ppo_safe_demo.gif', frames, fps=10)
print('Saved ppo_safe_demo.gif')

[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe
[INFO] Start Scenario Index: 0, Num Scenarios : 50
[INFO] Episode ended! Scenario Index: 23 Reason: crash vehicle 


Saved ppo_safe_demo.gif


In [12]:
import os

print("Generating demo GIFs for different scenarios...")

# Create output directory
output_dir = "outputs/demo"
os.makedirs(output_dir, exist_ok=True)

# Generate 9 GIFs (3 seeds × 3 traffic densities)
gif_index = generate_demo_suite(model, output_dir=output_dir)

print(f"\n🎬 Generated {len(gif_index)} GIFs in {output_dir}/")
for (seed, density), filepath in gif_index.items():
    print(f"   ✓ {os.path.basename(filepath)}")

print("\n✅ Step 4 Complete: GIFs generated!")
print("   Next: Run Cell 5 to view results or launch Gradio demo.")

[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector(), main_camera: MainCamera(1200, 900), dashboard: DashBoard()]
[INFO] Render Mode: onscreen
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe


Generating demo GIFs for different scenarios...
⚠️ close_engine not found, skipping...


[INFO] Start Scenario Index: 1008, Num Scenarios : 50


Generating GIF: seed=1008, density=0.05...


[INFO] Episode ended! Scenario Index: 1056 Reason: out_of_road.


...

GIF saved: outputs/demo/seed1012_density0.1.gif
⚠️ close_engine not found, skipping...


[INFO] Start Scenario Index: 1012, Num Scenarios : 50


Generating GIF: seed=1012, density=0.2...


[INFO] Episode ended! Scenario Index: 1041 Reason: out_of_road.


Episode ended at step 83
GIF saved: outputs/demo/seed1012_density0.2.gif

🎬 Generated 9 GIFs in outputs/demo/
   ✓ seed1008_density0.05.gif
   ✓ seed1008_density0.1.gif
   ✓ seed1008_density0.2.gif
   ✓ seed1010_density0.05.gif
   ✓ seed1010_density0.1.gif
   ✓ seed1010_density0.2.gif
   ✓ seed1012_density0.05.gif
   ✓ seed1012_density0.1.gif
   ✓ seed1012_density0.2.gif

✅ Step 4 Complete: GIFs generated!
   Next: Run Cell 5 to view results or launch Gradio demo.


## 6. Gradio Interactive Demo

In [ ]:
import gradio as gr

def show_gif(seed, traffic_density):
    key = (int(seed), float(traffic_density))
    return GIF_INDEX.get(key)


In [ ]:

demo = gr.Interface(
    fn=show_gif,
    inputs=[
        gr.Dropdown(
            choices=[1008, 1010, 1012],
            label="Scenario Seed",
            value=1000
        ),
        gr.Dropdown(
            choices=[0.05, 0.1, 0.2],
            label="Traffic Density",
            value=0.1
        ),
    ],
    outputs=gr.Image(type="filepath"),
    title="PPO MetaDrive Results Viewer",
    description="""
    This demo displays pre-generated PPO rollouts.
    MetaDrive runs offline; Gradio only visualizes results.
    """
)

demo.launch(share=True, debug=True)


In [ ]:
# Uncomment to launch the Gradio interface
# demo = create_gradio_demo(gif_index)
# demo.launch(share=True)

print("💡 To view the Gradio demo, uncomment the code above and run this cell.")
print(f"   GIFs are saved in: {os.path.abspath(output_dir)}")